<!-- nav -->
[← 7 · Bindings](07_Bindings.ipynb) · [Home](../../README.md) · [8.01 · Python Types →](08_01_Python_Types.ipynb)

# 8 · Sublanguages

Tutorials 3 to 7 built expressions, the terms a language computes values with. Every language also has grammars
that are not its expressions, written in an expression's place or within one: C writes a type name in a cast,
`(uint8_t)200`; TypeScript a type after `as`; Python, TypeScript and SystemVerilog a pattern where names are bound;
SystemVerilog a sequence in an assertion and a constraint in `randomize() with`; C and SystemVerilog a preprocessor
directive among them; TypeScript a regular expression in a literal, `/a+/g`, and Python and C++ one in a string. Each such grammar is a sublanguage, and each is a dialect of its own, of its language's family:
`Python Types`, `Python Patterns` and `Python RegExp`; `TypeScript Types`, `TypeScript Patterns` and
`TypeScript RegExp`; `C Types`, `Cpp Types`, `Cpp RegExp` and `C Preprocessor`; SystemVerilog's `Types`, `Patterns`, `Assertions`, `Constraints` and `Preprocessor`;
and, where C++'s regular expression names a grammar of POSIX's, POSIX's own, `POSIX BRE` and `POSIX ERE`. Its terms have
kinds, are validated, stored and printed as any dialect's, and are held two ways: in its own holder, an expression
whose root is one of its terms, and as subterms of the expression dialect that reads it, such as a cast's type.
No rule folds a sublanguage's kinds: they are modeled and printed, and stay in the residual, while the expressions that hold them fold as they did before.

Like the rest of Part I, it starts from `dialects()`, a store with every dialect installed and defined. `variable` writes a
name in the dialects whose names are the framework's `variable` (tutorial 4), as tutorials 5 and 6 do. Where a model is
Python's, Python also reads its text, with `Python3.Text.FromText`, and checks that it gives the same term (tutorial 3).
Regular expressions are not shown here, but every sublanguage, regular expressions among them, has a page of its
own after this one, which builds its kinds up from the smallest: the Python family's
[8.01 · Python Types](08_01_Python_Types.ipynb), [8.02 · Python Patterns](08_02_Python_Patterns.ipynb) and
[8.03 · Python RegExp](08_03_Python_RegExp.ipynb); the TypeScript family's
[8.04 · TypeScript Types](08_04_TypeScript_Types.ipynb), [8.05 · TypeScript Patterns](08_05_TypeScript_Patterns.ipynb)
and [8.06 · TypeScript RegExp](08_06_TypeScript_RegExp.ipynb); the C family's [8.07 · C Types](08_07_C_Types.ipynb),
[8.08 · C Preprocessor](08_08_C_Preprocessor.ipynb), [8.09 · Cpp Types](08_09_Cpp_Types.ipynb) and
[8.10 · Cpp RegExp](08_10_Cpp_RegExp.ipynb), with POSIX's [8.11 · POSIX BRE](08_11_POSIX_BRE.ipynb) and
[8.12 · POSIX ERE](08_12_POSIX_ERE.ipynb), which a C++ regular expression may hold; and SystemVerilog's
[8.13 · SystemVerilog Types](08_13_SystemVerilog_Types.ipynb),
[8.14 · SystemVerilog Patterns](08_14_SystemVerilog_Patterns.ipynb),
[8.15 · SystemVerilog Assertions](08_15_SystemVerilog_Assertions.ipynb),
[8.16 · SystemVerilog Constraints](08_16_SystemVerilog_Constraints.ipynb) and
[8.17 · SystemVerilog Preprocessor](08_17_SystemVerilog_Preprocessor.ipynb).

In [1]:
from mbse.Expressions.Python import Patterns as PythonPatterns, Python3, Types as PythonTypes
from mbse.Expressions.TypeScript import Patterns as TypeScriptPatterns, TypeScript5, Types as TypeScriptTypes
from mbse.Expressions.C import Expressions as C, Preprocessor as CPreprocessor, Types as CTypes
from mbse.Expressions.Cpp import Expressions as Cpp, Types as CppTypes
from mbse.Expressions.SystemVerilog import Assertions as SVAssertions, Constraints as SVConstraints, \
    Expressions as SystemVerilog, Patterns as SVPatterns, Preprocessor as SVPreprocessor, Types as SVTypes
from mbse.Expressions.Framework import Symbolics, Terms
from toolkit import dialects

store = dialects()
S = store.Expressions
amd64 = store.Instances.Expressions.SystemV.AMD64
variable = lambda name: lambda d: d.variable(lambda v: v.name(name))


def refused(simplify, expression):
    """What `simplify` says of `expression`: its value, a simplification's where its residual is a literal, or why it has
    none (`NotEvaluated`, a `ValueError`)."""
    try:
        result = simplify(expression)
        return result.evaluate() if hasattr(result, "evaluate") else result
    except ValueError as error:
        return str(error)

<a id="T08.1"></a>
## 08.1 · A type is another dialect's term

C's cast `(uint8_t)200` is C's `castexpression`: its `operand` is C's term, the literal `200`, and its `type` is a type
name, `C Types`' `typeid`, its specifiers (here one, the name `uint8_t`) and its abstract declarator (none). TypeScript's
`x as string` is TypeScript5's `tsasexpression`, its `typeAnnotation` a term of `TypeScript Types`, the keyword
`string`; SystemVerilog's `int'(x)` is its `castexpression`, its `type` a term of `SystemVerilog Types`. A form reads
each as it reads any term (tutorial 5): its arguments are the cast's terms, and `Terms.dialect_of` says which dialect's
union holds each one:

In [2]:
uint8 = lambda t: t.typeid(lambda i: i.specifiers([lambda s: s.namedtypespecifier(
    lambda n: n.name(lambda m: m.identifier(lambda d: d.spelling("uint8_t"))))]))
with store.transaction():
    casts = {
        "C": (C.Text, S.C.Expressions.Expression(lambda e: e.root(lambda u: u.castexpression(lambda r: r
            .type(uint8).operand(lambda o: o.literal(lambda l: l.value(lambda v: v.as_scalar(200)))))).abi([amd64]))),
        "TypeScript5": (TypeScript5.Text, S.TypeScript.TypeScript5.Expression(lambda e: e.root(
            lambda u: u.tsasexpression(lambda r: r
                .expression(lambda x: x.identifier(lambda i: i.name("x")))
                .typeAnnotation(lambda t: t.tsstringkeyword(lambda k: k)))))),
        "SystemVerilog": (SystemVerilog.Text, S.SystemVerilog.Expressions.Expression(lambda e: e.root(
            lambda u: u.castexpression(lambda r: r
                .type(lambda t: t.integeratomtype(lambda a: a.keyword("int")))
                .value(lambda v: v.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling("x"))))))))),
    }
for language, (text, cast) in casts.items():
    form = Terms.form(Terms.term(cast))
    print(f"{language:14}", f"{text.ToText(cast):17}", form.kind)
    for argument in form.arguments:
        print(" " * 15, f"{Terms.kind_of(argument)[0]:17}", Terms.dialect_of(argument))
assert [text.ToText(cast) for text, cast in casts.values()] == ["(uint8_t)200", "x as string", "int'(x)"]

C              (uint8_t)200      castexpression
                typeid            Expressions.C.Types.Term
                literal           Expressions.C.Expressions.Term
TypeScript5    x as string       tsasexpression
                identifier        Expressions.TypeScript.TypeScript5.Term
                tsstringkeyword   Expressions.TypeScript.Types.Term
SystemVerilog  int'(x)           castexpression
                integeratomtype   Expressions.SystemVerilog.Types.Term
                nameexpression    Expressions.SystemVerilog.Expressions.Term


Each host reads its sublanguages' terms through the dialects it reads, `reads()`: C reads `C Types`' and
`C Preprocessor`'s, TypeScript5 `TypeScript Types`', `TypeScript Patterns`' and
`TypeScript RegExp`'s, SystemVerilog its five sublanguages'.
`owner` names the dialect whose kind a term is. So the host validates, infers and simplifies a cast with its type in
it, as it did before the types were dialects of their own: C's rules read the type to convert `200`, and infers
the cast's domain from it, the ABI's `uint8_t`; SystemVerilog's converts `x` to an `int`; TypeScript5 folds no `as`, as tutorial 3 said it folds its literals alone:

In [3]:
hosts = {"C": (C.Expressions.DIALECT, C.Simplifiers.OfAny),
         "TypeScript5": (TypeScript5.Expressions.DIALECT, TypeScript5.Simplifiers.OfAny),
         "SystemVerilog": (SystemVerilog.Expressions.DIALECT, SystemVerilog.Simplifiers.OfAny)}
for language, (text, cast) in casts.items():
    dialect, simplify = hosts[language]
    type_ = next(a for a in Terms.form(Terms.term(cast)).arguments if Terms.dialect_of(a) != dialect.union)
    print(f"{language:14}", "reads", ", ".join(d.name() for d in dialect.reads()))
    print(" " * 14, "its type's owner:", dialect.owner(type_).name(),
          "| problems:", ", ".join(dialect.validate(cast, {"x"})) or "none",
          "| x = 5:", refused(lambda c: simplify(c, {"x": 5}), cast))
print("C infers the cast's domain:", C.Expressions.DIALECT.infer(casts["C"][1]).name())

C              reads C Types, C Preprocessor
               its type's owner: C Types | problems: none | x = 5: the root is not a literal: its value has no literal
TypeScript5    reads TypeScript Types, TypeScript Patterns, TypeScript RegExp
               its type's owner: TypeScript Types | problems: none | x = 5: the root is not a literal: TypeScript5 tsasexpression is not simplified
SystemVerilog  reads SystemVerilog Types, SystemVerilog Patterns, SystemVerilog Assertions, SystemVerilog Constraints, SystemVerilog Preprocessor
               its type's owner: SystemVerilog Types | problems: none | x = 5: 32'sb00000000000000000000000000000101
C infers the cast's domain: uint8_t


<a id="T08.2"></a>
## 08.2 · A binding target is a pattern

Where TypeScript binds names by destructuring, it writes a pattern: the arrow `([a, b = 1]) => a + b` takes one
parameter, the array pattern `[a, b = 1]`, a term of `TypeScript Patterns` (its `arraypattern`, whose second element is
an `assignmentpattern`, a name and its default). SystemVerilog's `matches` compares a value with a pattern of
`SystemVerilog Patterns`, here the structure pattern `'{.a, .b}`, whose variables it binds. A pattern binds names, but
binds them through what holds it: none of its kinds is a binder of its own (tutorial 7). Each pattern dialect gives the
names a pattern binds, TypeScript's `bound_names` and SystemVerilog's `bound_names`; the arrow binds its parameters'
names in its body, and a conditional binds its `matches`' variables in its consequence, so the only free name left is
`v`, the value matched:

In [4]:
with store.transaction():
    arrow = S.TypeScript.TypeScript5.Expression(lambda e: e.root(lambda u: u.arrowfunctionexpression(lambda a: a
        .params([lambda p: p.arraypattern(lambda r: r.elements([
            lambda d: d.identifier(lambda i: i.name("a")),
            lambda d: d.assignmentpattern(lambda s: s.left(lambda d: d.identifier(lambda i: i.name("b"))).right(
                lambda o: o.literal(lambda l: l.value(lambda v: v.as_scalar(1.0)).text("1"))))]))])
        .body(lambda b: b.binaryexpression(lambda o: o
            .left(lambda d: d.identifier(lambda i: i.name("a"))).operator("+")
            .right(lambda d: d.identifier(lambda i: i.name("b"))))))))
    matches = S.SystemVerilog.Expressions.Expression(lambda e: e.root(lambda u: u.conditionalexpression(lambda c: c
        .condition(lambda d: d.matchesexpression(lambda m: m
            .value(lambda d: d.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling("v")))))
            .pattern(lambda p: p.structurepattern(lambda s: s.items([
                lambda i: i.variablepattern(lambda r: r.name(lambda n: n.spelling("a"))),
                lambda i: i.variablepattern(lambda r: r.name(lambda n: n.spelling("b")))])))))
        .consequence(lambda d: d.binaryexpression(lambda b: b
            .left(lambda d: d.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling("a")))))
            .operator("+")
            .right(lambda d: d.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling("b")))))))
        .alternative(lambda d: d.literal(lambda l: l.value(lambda v: v.as_scalar(0)))))))
parameter = Terms.form(Terms.term(arrow)).arguments[-1]  # after Let's empty subterms and its body: its one parameter
pattern = Terms.form(Terms.form(Terms.term(matches)).arguments[0]).arguments[1]  # the condition's pattern
for text, dialect, expression, target, names in [
        (TypeScript5.Text, TypeScript5.Expressions.DIALECT, arrow, parameter, TypeScriptPatterns.Expressions.bound_names),
        (SystemVerilog.Text, SystemVerilog.Expressions.DIALECT, matches, pattern, SVPatterns.Expressions.bound_names)]:
    print(text.ToText(expression))
    print(" ", f"{text.ToText(target):9}", Terms.kind_of(target)[0], "of", Terms.dialect_of(target))
    print("  binds", ", ".join(sorted(names(target))), "| free:", ", ".join(sorted(Symbolics.free(expression, dialect))) or "none")
assert TypeScript5.Text.ToText(arrow) == "([a, b = 1]) => a + b"
assert Symbolics.free(arrow, TypeScript5.Expressions.DIALECT) == set()
assert SystemVerilog.Text.ToText(matches) == "v matches '{.a, .b} ? a + b : 0"
assert Symbolics.free(matches, SystemVerilog.Expressions.DIALECT) == {"v"}

([a, b = 1]) => a + b
  [a, b = 1] arraypattern of Expressions.TypeScript.Patterns.Term
  binds a, b | free: none
v matches '{.a, .b} ? a + b : 0
  '{.a, .b} structurepattern of Expressions.SystemVerilog.Patterns.Term
  binds a, b | free: v


Python's patterns are those of its `match` statement, `Python Patterns`: `case [a, *rest]:` matches a sequence and binds
`a` and `rest`. A `match` is a statement, which no expression holds, so only the dialect's own holder holds a pattern
here (mbse-programs' `case` holds one too). Python's other binding targets are its expressions as `ast` has them, not
patterns: a comprehension's target, `a, b` in `[a + b for a, b in pairs]`, is Python3's `tuple` of `variable`s, held
by its `comprehension` helper (tutorial 7):

In [5]:
with store.transaction():
    sequence = S.Python.Patterns.Expression(lambda e: e.root(lambda u: u.matchsequence(lambda s: s.delimiters("[]").patterns([
        lambda p: p.matchas(lambda m: m.name("a")), lambda p: p.matchstar(lambda m: m.name("rest"))]))))
    comprehension = S.Python.Python3.Expression(lambda e: e.root(lambda u: u.listcomp(lambda c: c
        .body(lambda d: d.binop(lambda b: b.left(variable("a")).op("+").right(variable("b"))))
        .generators([lambda g: g.target(lambda t: t.tuple(lambda p: p.elts([variable("a"), variable("b")])))
                     .iter(variable("pairs"))]))))
    read = S.Python.Python3.Expression(lambda e: e.root(Python3.Text.FromText("[a + b for a, b in pairs]")))
print(f"{PythonPatterns.Text.ToText(sequence):11}", Terms.kind_of(Terms.term(sequence))[0], "of", Terms.dialect_of(Terms.term(sequence)))
print("  binds", ", ".join(sorted(PythonPatterns.Expressions.bound_names(Terms.term(sequence)))))
print(Python3.Text.ToText(comprehension), "|", Terms.kind_of(Terms.term(comprehension))[0], "of",
      Terms.dialect_of(Terms.term(comprehension)))
generator = Terms.form(Terms.term(comprehension)).attributes["generators"][0]  # a helper, a Terms.Part (tutorial 6)
print(" ", generator.kind, "| its target:", generator.attributes["target"].kind, "of", generator.attributes["target"].size,
      "terms, each", Terms.dialect_of(Terms.form(Terms.term(comprehension)).arguments[-1]))
assert PythonPatterns.Text.ToText(sequence) == "[a, *rest]"
assert PythonPatterns.Expressions.bound_names(Terms.term(sequence)) == {"a", "rest"}
assert Terms.same(read, comprehension)  # Python reads its own text into the same term

[a, *rest]  matchsequence of Expressions.Python.Patterns.Term
  binds a, rest
[a + b for a, b in pairs] | listcomp of Expressions.Python.Python3.Term
  comprehension | its target: tuple of 2 terms, each Expressions.Python.Python3.Term


<a id="T08.3"></a>
## 08.3 · Each sublanguage a dialect, with its holder

A sublanguage is a dialect like any other (tutorial 1): a singleton of its own, named with its family
(`C Types`, tag `c_types`), its `Term` a union of its kinds, its kinds' editions recorded as data. And it has a holder
of its own, `S.<Family>.<Sublanguage>.Expression`, which refines the framework's `Expression` as every holder does
(tutorial 3): its `root` is a term of its dialect, so a type, a pattern, a sequence, a constraint or a directive can be
written down, stored and printed alone. A C family's holder names its ABI, as C's does (tutorial 2).

Each sublanguage's `Term` lists, besides its own kinds, what may stand where they do: Python Types' lists all of
Python3's branches (a type expression is any expression), SystemVerilog's sublanguages list its `Expressions`'
branches, C Types' lists the preprocessor's directives (an enumeration's items may be). So `T: list[int] | None = int`, a type parameter of
`Python Types`, holds a type expression of Python3's kinds as its bound, which Python reads from its text:

In [6]:
with store.transaction():
    type_parameter = S.Python.Types.Expression(lambda e: e.root(lambda u: u.typevar(lambda t: t.name("T")
        .bound(lambda b: b.binop(lambda o: o
            .left(lambda l: l.subscript(lambda s: s.value(variable("list")).slice(variable("int"))))
            .op("|").right(lambda r: r.constant(lambda k: k.singleton("None").text("None")))))
        .default_value(variable("int")))))
    read = S.Python.Types.Expression(lambda e: e.root(lambda u: u.typevar(lambda t: t.name("T")
        .bound(Python3.Text.FromText("list[int] | None")).default_value(Python3.Text.FromText("int")))))
    sublanguages = {
        "Python Types": (PythonTypes, Python3, type_parameter),
        "Python Patterns": (PythonPatterns, Python3, sequence),
        "TypeScript Types": (TypeScriptTypes, TypeScript5, S.TypeScript.Types.Expression(lambda e: e.root(
            lambda u: u.tsuniontype(lambda t: t.types([lambda k: k.tsstringkeyword(lambda h: h),
                                                       lambda k: k.tsnumberkeyword(lambda h: h)]))))),
        "TypeScript Patterns": (TypeScriptPatterns, TypeScript5, S.TypeScript.Patterns.Expression(lambda e: e.root(
            lambda u: u.arraypattern(lambda r: r.elements([
                lambda d: d.identifier(lambda i: i.name("a")),
                lambda d: d.restelement(lambda s: s.argument(lambda d: d.identifier(lambda i: i.name("rest"))))]))))),
        "C Types": (CTypes, C, S.C.Types.Expression(lambda e: e.root(lambda u: u.typeid(lambda i: i
            .specifiers([lambda s: s.primitivetypespecifier(lambda p: p.keyword("unsigned")),
                         lambda s: s.primitivetypespecifier(lambda p: p.keyword("char"))])
            .declarator(lambda d: d.pointerdeclarator(lambda p: p)))).abi([amd64]))),
        "Cpp Types": (CppTypes, Cpp, S.Cpp.Types.Expression(lambda e: e.root(lambda u: u.typeid(lambda i: i
            .specifiers([lambda s: s.primitivetypespecifier(lambda p: p.keyword("unsigned")),
                         lambda s: s.primitivetypespecifier(lambda p: p.keyword("long"))]))).abi([amd64]))),
        "C Preprocessor": (CPreprocessor, C, S.C.Preprocessor.Expression(lambda e: e.root(
            lambda u: u.definedirective(lambda d: d.name(lambda n: n.spelling("WIDTH")).replacement("8"))).abi([amd64]))),
        "SystemVerilog Types": (SVTypes, SystemVerilog, S.SystemVerilog.Types.Expression(lambda e: e.root(
            lambda u: u.integervectortype(lambda v: v.keyword("bit").dimensions([lambda d: d.rangedimension(lambda r: r
                .left(lambda l: l.literal(lambda li: li.value(lambda v: v.as_scalar(3))))
                .right(lambda l: l.literal(lambda li: li.value(lambda v: v.as_scalar(0)))))]))))),
        "SystemVerilog Patterns": (SVPatterns, SystemVerilog, S.SystemVerilog.Patterns.Expression(lambda e: e.root(
            lambda u: u.taggedpattern(lambda t: t.member(lambda m: m.spelling("Valid"))
                .pattern(lambda p: p.variablepattern(lambda r: r.name(lambda n: n.spelling("x")))))))),
        "SystemVerilog Assertions": (SVAssertions, SystemVerilog, S.SystemVerilog.Assertions.Expression(lambda e: e.root(
            lambda u: u.delaysequence(lambda s: s
                .first(lambda d: d.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling("a")))))
                .steps([lambda t: t.delay(lambda d: d.literal(lambda l: l.value(lambda v: v.as_scalar(1))))
                        .sequence(lambda d: d.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling("b")))))]))))),
        "SystemVerilog Constraints": (SVConstraints, SystemVerilog, S.SystemVerilog.Constraints.Expression(lambda e: e.root(
            lambda u: u.expressionconstraint(lambda c: c.expression(lambda d: d.insideexpression(lambda i: i
                .value(lambda d: d.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling("x")))))
                .set([lambda r: r.valuerange(lambda v: v
                    .left(lambda l: l.literal(lambda li: li.value(lambda n: n.as_scalar(0))))
                    .right(lambda l: l.literal(lambda li: li.value(lambda n: n.as_scalar(9)))))]))))))),
        "SystemVerilog Preprocessor": (SVPreprocessor, SystemVerilog, S.SystemVerilog.Preprocessor.Expression(lambda e: e.root(
            lambda u: u.definedirective(lambda d: d.name(lambda n: n.spelling("WIDTH")).body("8"))))),
    }
assert Terms.same(read, type_parameter)  # Python reads a type expression's text: it is Python3's
for name, (package, family, holder) in sublanguages.items():
    dialect = package.Expressions.DIALECT
    schema = store.registered(f"Expressions.{dialect.path}.Expression")
    assert dialect.name() == name and Terms.is_expression(schema) and Terms.dialect_of(Terms.term(holder)) == dialect.union
    print(f"{name:27}", f"{dialect.tag:27}", f"{package.Text.ToText(holder):26}", Terms.kind_of(Terms.term(holder))[0])
assert [package.Text.ToText(holder) for package, _, holder in sublanguages.values()] == [
    "T: list[int] | None = int", "[a, *rest]", "string | number", "[a, ...rest]", "unsigned char*", "unsigned long",
    "#define WIDTH 8", "bit [3:0]", "tagged Valid .x", "a ##1 b", "x inside {[0:9]};", "`define WIDTH 8"]

Python Types                python_types                T: list[int] | None = int  typevar
Python Patterns             python_patterns             [a, *rest]                 matchsequence
TypeScript Types            typescript_types            string | number            tsuniontype
TypeScript Patterns         typescript_patterns         [a, ...rest]               arraypattern
C Types                     c_types                     unsigned char*             typeid
Cpp Types                   cpp_types                   unsigned long              typeid
C Preprocessor              c_preprocessor              #define WIDTH 8            definedirective
SystemVerilog Types         systemverilog_types         bit [3:0]                  integervectortype
SystemVerilog Patterns      systemverilog_patterns      tagged Valid .x            taggedpattern
SystemVerilog Assertions    systemverilog_assertions    a ##1 b                    delaysequence
SystemVerilog Constraints   systemverilog_const

Its expression dialect holds a sublanguage's terms two ways. A subterm of one of its kinds may be of the sublanguage's
`Term`: a cast's type, an arrow's parameters, a `matches`' pattern, a call's arguments (a sequence or a property may
be one), `randomize() with`'s items. And its own `Term` may list a few of the sublanguage's kinds, where its grammar
lets them stand as expressions: TypeScript5's lists the patterns an assignment's target may be, C's and C++'s list
`typeid` (`sizeof(int)`) and the preprocessor's `defined(X)`, SystemVerilog's the preprocessor's macro usage. Python3's
kinds hold neither: a type expression is Python3's own, and a parameter's annotation, which mbse-programs' function
definitions hold, is the helper `arg`'s; a pattern only `match` holds. The properties that hold such a subterm are read from the
store. And each family's rules fold none of the sublanguage's kinds: each stays in the residual, named by the dialect that owns it:

In [7]:
simplifiers = {Python3: Python3.Simplifiers.OfAny, TypeScript5: TypeScript5.Simplifiers.OfAny, C: C.Simplifiers.OfAny,
              Cpp: Cpp.Simplifiers.OfAny, SystemVerilog: SystemVerilog.Simplifiers.OfAny}
for name, (package, family, holder) in sublanguages.items():
    union, own = store.registered(package.Expressions.DIALECT.union), package.Expressions.TAGS
    host = store.registered(family.Expressions.DIALECT.union).branches
    properties = [f"{b.name}.{p}" for b in host if b.name not in own for p, prop in b.schema.properties.items()
                 if union in (prop.schema, getattr(prop.schema, "element", None))]
    listed = [b.name for b in host if b.name in own]
    print(f"{name:27}", family.Expressions.DIALECT.name())
    print(" " * 28, "properties:", ", ".join(properties) or "none", "| its kinds listed:", ", ".join(listed) or "none")
    print(" " * 28, refused(simplifiers[family], holder))

Python Types                Python3
                             properties: none | its kinds listed: none
                             the root is not a literal: Python Types typevar is not simplified
Python Patterns             Python3
                             properties: none | its kinds listed: none
                             the root is not a literal: Python Patterns matchsequence is not simplified
TypeScript Types            TypeScript5
                             properties: tsasexpression.typeAnnotation, tssatisfiesexpression.typeAnnotation, tstypeassertion.typeAnnotation | its kinds listed: none
                             the root is not a literal: TypeScript Types tsuniontype is not simplified
TypeScript Patterns         TypeScript5
                             properties: functionexpression.params, arrowfunctionexpression.params, tsemptybodyfunctionexpression.params, assignmentexpression.left | its kinds listed: arraypattern, objectpattern, assignmentpattern
       

<a id="T08.4"></a>
## 08.4 · SystemVerilog's assertions and constraints

SystemVerilog Assertions describe behavior over clock cycles: a sequence, `a ##1 b` (`a`, then `b` one cycle later), a
property built of sequences, `a |=> b` (whenever `a`, `b` in the next cycle), and a property's spec, its clock and when
it is disabled, `@(posedge clk) disable iff (rst)`. Each is a kind of `SystemVerilog Assertions`, held by its holder,
or by a call's arguments, where a named sequence or property is given one; the statement that asserts a property,
`assert property (...)`, is mbse-programs'.
SystemVerilog Constraints say what values a randomized object may take: `x inside {[0:9]}`, an expression held as a
constraint, and `x dist {0 := 1, [1:9] :/ 9}`, a distribution of its values, weighted, each a kind of `SystemVerilog
Constraints`, held by its holder or by `randomize() with`'s items. They are modeled and printed, and not simplified:
SystemVerilog's rules neither simulate clock cycles nor solves constraints:

In [8]:
with store.transaction():
    handshake = S.SystemVerilog.Assertions.Expression(lambda e: e.root(lambda u: u.propertyspec(lambda s: s
        .clock(lambda k: k.events([lambda v: v.edge("posedge").expression(
            lambda d: d.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling("clk")))))]))
        .disable(lambda d: d.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling("rst")))))
        .property(lambda p: p.implicationproperty(lambda i: i
            .antecedent(lambda d: d.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling("a")))))
            .operator("|=>")
            .consequent(lambda d: d.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling("b"))))))))))
    called = S.SystemVerilog.Expressions.Expression(lambda e: e.root(lambda u: u.callexpression(lambda c: c
        .callee(lambda d: d.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling("f")))))
        .arguments([lambda t: t.delaysequence(lambda s: s
            .first(lambda d: d.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling("a")))))
            .steps([lambda p: p.delay(lambda d: d.literal(lambda l: l.value(lambda v: v.as_scalar(1))))
                    .sequence(lambda d: d.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling("b")))))]))]))))
    randomized = S.SystemVerilog.Expressions.Expression(lambda e: e.root(lambda u: u.randomizewithexpression(lambda r: r
        .call(lambda c: c.callexpression(lambda m: m.callee(lambda d: d.memberexpression(lambda b: b
            .value(lambda d: d.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling("p")))))
            .member(lambda i: i.spelling("randomize")))).arguments([])))
        .items([
            lambda t: t.expressionconstraint(lambda c: c.expression(lambda d: d.insideexpression(lambda i: i
                .value(lambda d: d.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling("x")))))
                .set([lambda r: r.valuerange(lambda v: v.left(lambda l: l.literal(lambda li: li.value(lambda n: n.as_scalar(0))))
                                                      .right(lambda l: l.literal(lambda li: li.value(lambda n: n.as_scalar(9)))))])))),
            lambda t: t.expressionconstraint(lambda c: c.expression(lambda d: d.distexpression(lambda i: i
                .value(lambda d: d.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling("x")))))
                .items([lambda w: w.value(lambda l: l.literal(lambda li: li.value(lambda n: n.as_scalar(0))))
                        .operator(":=").weight(lambda l: l.literal(lambda li: li.value(lambda n: n.as_scalar(1)))),
                        lambda w: w.value(lambda r: r.valuerange(lambda v: v
                            .left(lambda l: l.literal(lambda li: li.value(lambda n: n.as_scalar(1))))
                            .right(lambda l: l.literal(lambda li: li.value(lambda n: n.as_scalar(9))))))
                        .operator(":/").weight(lambda l: l.literal(lambda li: li.value(lambda n: n.as_scalar(9))))]))))]))))
dialect, simplify = SystemVerilog.Expressions.DIALECT, SystemVerilog.Simplifiers.OfAny
for expression in (handshake, called, randomized):
    print(SystemVerilog.Text.ToText(expression))
    print(" ", ", ".join(f"{Terms.kind_of(a)[0]} of {dialect.owner(a).name()}"
                         for a in Terms.form(Terms.term(expression)).arguments if a is not None))
    print("  free:", ", ".join(sorted(Symbolics.free(expression, dialect))))
print(refused(simplify, handshake), "|", refused(simplify, randomized))
assert [SystemVerilog.Text.ToText(e) for e in (handshake, called, randomized)] == [
    "@(posedge clk) disable iff (rst) a |=> b", "f(a ##1 b)",
    "p.randomize() with { x inside {[0:9]}; x dist {0 := 1, [1:9] :/ 9}; }"]

@(posedge clk) disable iff (rst) a |=> b
  nameexpression of SystemVerilog, implicationproperty of SystemVerilog Assertions, nameexpression of SystemVerilog
  free: a, b, clk, rst
f(a ##1 b)
  nameexpression of SystemVerilog, delaysequence of SystemVerilog Assertions
  free: a, b, f
p.randomize() with { x inside {[0:9]}; x dist {0 := 1, [1:9] :/ 9}; }
  callexpression of SystemVerilog, expressionconstraint of SystemVerilog Constraints, expressionconstraint of SystemVerilog Constraints
  free: p, x
the root is not a literal: SystemVerilog Assertions propertyspec is not simplified | the root is not a literal: SystemVerilog randomizewithexpression is not simplified


<a id="T08.5"></a>
## 08.5 · One printer per family

A family prints its dialects with one printer, as mbse-programs prints a program: a host's printer prints the
sublanguages' terms it holds (08.1 to 08.4), and each sublanguage's `Text.ToText` prints through it. Python's,
TypeScript's and SystemVerilog's sublanguages give their family's `ToText` itself; C's and C++'s give one of their own,
which calls the C family's printer naming their dialect. So a tree that crosses dialects prints as one text, whichever
holder holds it. C's enumeration below is a `C Types` term whose items are its enumerators and a `C Preprocessor`
directive, `#ifdef WIDE`, whose items are `C Types`' enumerators again; the printer writes the directive at its line's
start, and the body's enumerators on lines of their own:

In [9]:
for family, subs in [(Python3, [PythonTypes, PythonPatterns]), (TypeScript5, [TypeScriptTypes, TypeScriptPatterns]),
                     (C, [CTypes, CPreprocessor]), (Cpp, [CppTypes]),
                     (SystemVerilog, [SVTypes, SVPatterns, SVAssertions, SVConstraints, SVPreprocessor])]:
    print(f"{family.Expressions.DIALECT.name():14}",
          *(f"{s.Expressions.DIALECT.name()}: {s.Text.ToText is family.Text.ToText}" for s in subs))
with store.transaction():
    color = S.C.Types.Expression(lambda e: e.root(lambda u: u.typeid(lambda i: i.specifiers([
        lambda s: s.enumspecifier(lambda n: n.key("enum").name(lambda m: m.identifier(lambda d: d.spelling("Color"))).body(True)
            .enumerators([
                lambda t: t.enumerator(lambda r: r.name(lambda d: d.spelling("RED"))
                    .value(lambda v: v.literal(lambda l: l.value(lambda n: n.as_scalar(1))))),
                lambda t: t.ifdefdirective(lambda d: d.name(lambda n: n.spelling("WIDE")).items([
                    lambda t: t.c_types(lambda u: u.enumerator(lambda r: r.name(lambda d: d.spelling("GREEN"))))])),
                lambda t: t.enumerator(lambda r: r.name(lambda d: d.spelling("BLUE")))]))]))).abi([amd64]))
print(CTypes.Text.ToText(color))
directive = Terms.form(Terms.term(color)).arguments[2]
print(Terms.kind_of(directive)[0], "of", Terms.dialect_of(directive), "|", CTypes.Expressions.DIALECT.owner(directive).name())
assert CTypes.Text.ToText(color) == C.Text.ToText(color) == (
    "enum Color {\n    RED = 1,\n#ifdef WIDE\n    GREEN,\n#endif\n    BLUE\n}")

Python3        Python Types: True Python Patterns: True
TypeScript5    TypeScript Types: True TypeScript Patterns: True
C              C Types: False C Preprocessor: False
Cpp            Cpp Types: False
SystemVerilog  SystemVerilog Types: True SystemVerilog Patterns: True SystemVerilog Assertions: True SystemVerilog Constraints: True SystemVerilog Preprocessor: True
enum Color {
    RED = 1,
#ifdef WIDE
    GREEN,
#endif
    BLUE
}
ifdefdirective of Expressions.C.Types.Term | C Preprocessor


## What you learned

- **A sublanguage is a dialect of its own**, of its language's family: its kinds, its `Term`, its singleton named with
  its family, its editions as data. Python's types and patterns, TypeScript's types and patterns, C's and C++'s types
  and their preprocessor, and SystemVerilog's types, patterns, assertions, constraints and preprocessor are twelve; with the regular
  expressions of Python, TypeScript and C++, and POSIX's two that C++ reads, seventeen.
- **A type or a pattern is a term**, of its sublanguage, where its host's grammar puts one: a cast's type, an arrow's
  parameters, a `matches`' pattern, a call's argument, `randomize() with`'s items. The host reads it through the
  dialects it reads, and `owner` names the dialect it is of.
- **Each has a holder of its own**, an expression whose root is one of its terms, so a type, a pattern, a sequence, a
  constraint or a directive is written down and stored alone.
- **A pattern binds through what holds it**: an arrow its parameters, a `matches` its variables, a `case` its captures.
- **None is folded**: each family's rules fold none of a sublanguage's kinds, which stay in the residual, named by the
  dialect that owns them, while a host's own kinds keep folding as they did (C's cast reads its type).
- **One printer per family** prints a tree that crosses its dialects as one text.

Next, a page for each sublanguage, starting with [8.01 · Python Types](08_01_Python_Types.ipynb); after them,
Part II puts expressions to work as constraints, starting with case study 9.

<!-- nav -->
[← 7 · Bindings](07_Bindings.ipynb) · [Home](../../README.md) · [8.01 · Python Types →](08_01_Python_Types.ipynb)